# Boundary Element Model - Build Model

The Boundary Element Model (BEM) defines the different compartments of the head (brain, skull, and scalp) by generating three distinct surfaces:
- the inner skull
- the outer skull
- the outer scalp.

The BEM is constructed by combining these surfaces, with specified electrical conductivities between the layers. These conductivities are typically assumed to be homogeneous and isotropic within each compartment. <br>

We first generate the surfaces by segmenting the T1-weighted MRI using FreeSurfer’s watershed algorithm. Ideally, the resulting surfaces should be concentric; however, small inaccuracies can occur and must be corrected before proceeding. This correction is performed using our `correct_skull` function. Finally, the corrected surfaces are assembled by assigning the electrical conductivities that define each inter-compartment region.

## 🧪 - Experiment setup

In [1]:
# imports
import mindmap as mm
import mne

# readers
dm = mm.io.TutoManager("../../config.yml")
sr = dm.get_subject("spike11")

## 🧬 - Data Processing

**Build layers**
MNE-Python’s watershed implementation is a wrapper for the FreeSurfer watershed algorithm, which utilizes the T1 image within the FreeSurfer directory to generate the following outputs:
- `outer_skin.surf`: The outer skin (scalp) surface.
- `outer_skull_raw.surf`: The outer skull surface.
- `inner_skull.surf`: The final inner skull surface.
- `brain.surf`: A copy of the brain surface.
- `inner_skull_raw.surf`: If the inner skull extends beyond the outer skull (a known watershed bug), this file preserves the original, uncorrected surface before vertices are adjusted.
- `inner_skull_changes.vtk`: A visualization file showing the spatial differences between the raw and corrected inner skull surfaces.

In [4]:
# Build layers from T1
print("\033[95mBuild BEM surfaces...\033[0m")
mne.bem.make_watershed_bem(
    subject      = f"sub-{sr.sub}",
    subjects_dir = dm.roots["fs"],
    volume       = "T1",
    overwrite    = True,
)

# correct surfaces
print("\033[95mCorrect surfaces...\033[0m")
mm.anatomy.forward.correct_skull(
    subject     = f"sub-{sr.sub}",
    offset      = 2,                    # minimum skull thickness to apply in [mm]
    fs_dir      = dm.roots["fs"],
    save_arrows = True,
)

Build BEM surfaces...

Running mri_watershed for BEM segmentation with the following parameters:

Results dir = /home/nino/Code/mindmap/data/derivatives/freesurfer/sub-spike11/bem/watershed
Command = mri_watershed -useSRAS -surf /home/nino/Code/mindmap/data/derivatives/freesurfer/sub-spike11/bem/watershed/sub-spike11 /home/nino/Code/mindmap/data/derivatives/freesurfer/sub-spike11/mri/T1.mgz /home/nino/Code/mindmap/data/derivatives/freesurfer/sub-spike11/bem/watershed/ws.mgz

Running subprocess: mri_watershed -useSRAS -surf /home/nino/Code/mindmap/data/derivatives/freesurfer/sub-spike11/bem/watershed/sub-spike11 /home/nino/Code/mindmap/data/derivatives/freesurfer/sub-spike11/mri/T1.mgz /home/nino/Code/mindmap/data/derivatives/freesurfer/sub-spike11/bem/watershed/ws.mgz

Mode:          use surfaceRAS to save surface vertex positions
Mode:          Saving out BEM surfaces

*********************************************************
The input file is /home/nino/Code/mindmap/data/derivatives/

**Assemble model**

In [5]:
# assemble
model = mne.make_bem_model(
        subject      = f"sub-{sr.sub}",
        subjects_dir = dm.config["DIR_FS"],
        conductivity = [0.3, 0.006, 0.3],    # conductivities (brain, skull, skin)
        ico          = 4,                    # mesh resolution (needed to simplify computations)
)

bem  = mne.make_bem_solution(model)

Creating the BEM geometry...
Going from 5th to 4th subdivision of an icosahedron (n_tri: 20480 -> 5120)
Going from 5th to 4th subdivision of an icosahedron (n_tri: 20480 -> 5120)
Going from 5th to 4th subdivision of an icosahedron (n_tri: 20480 -> 5120)
outer skin  CM is  -0.09 -17.54  14.92 mm
outer skull CM is  -0.09 -17.34  14.49 mm
inner skull CM is  -0.21 -21.38  22.36 mm
Checking that surface outer skull is inside surface outer skin  ...
Checking that surface inner skull is inside surface outer skull ...
Checking distance between outer skin  and outer skull surfaces...
Minimum distance between the outer skin  and outer skull surfaces is approximately    1.9 mm
Checking distance between outer skull and inner skull surfaces...
Minimum distance between the outer skull and inner skull surfaces is approximately    2.0 mm
Surfaces passed the basic topology checks.
Complete.

Three-layer model surfaces loaded.
Computing the linear collocation solution...
    Matrix coefficients...
     

## 📊 - Visualization

BEM layers

In [6]:
mm.viz.plot_bem(dm.roots["fs"], sr.sub)